In [ ]:
# Kaggle Pipeline Overview: Fine-Tuning YOLO26 on Class-Specific Lesions
# 
# Context & Strategy:
# - Scenario: Fine-tuning a pre-trained YOLO26 model (previously trained on a generic 'acne' category) 
#   using 200 newly annotated images featuring specific lesion categories (papule, pustule, comedone).
#   Legacy annotations (800 images) are preserved in their original state.
# 
# Key Technical Decisions:
# - Transfer Learning: Initialize weights from the base acne model to accelerate convergence and preserve learned visual features.
# - Target Classes: Fine-tune the architecture to resolve granular lesion types (papule, pustule, comedone).
# - Isolated Training Set: Train strictly on the 200 multi-class annotated samples to prevent label noise from coarse legacy data.
# - Baseline Fine-Tuning: Perform a standard supervised training run prior to hyperparameter optimization (e.g., Optuna) 
#   to establish performance baselines while conserving compute resources.
# - Independent Evaluation: Reserve a distinct test partition to evaluate generalization on unseen images.
#
# Prerequisite:
# - The 200-image dataset must already be split into 'train', 'valid', and 'test' subdirectories with standard YOLO format text labels.

# Kaggle Workspace Setup:
# Configure the following items as Notebook Inputs:
# 1. Dataset containing the 200 annotated images.
# 2. Checkpoint artifact from the base model (best.pt or last.pt).
#
# Expected Dataset Directory Structure:
# acne_200/
# ├── train/
# │   ├── images/
# │   └── labels/
# ├── valid/
# │   ├── images/
# │   └── labels/
# └── test/
#     ├── images/
#     └── labels/

# Label Indexing:
# Ensure each image has a corresponding .txt annotation file. 
# Class index mapping:
# 0 = papule
# 1 = pustule
# 2 = comedone
#
# If class IDs differ in your dataset, update the CLASS_NAMES dictionary in the configuration section rather than modifying label files.


# Setup & Environment Initialization
# Execute this cell first. Ensure path constants match your workspace paths in /kaggle/input.

!pip install -q -U ultralytics pyyaml

import os
import gc
import yaml
import shutil
import random
import zipfile
from pathlib import Path
from datetime import datetime

import torch
from ultralytics import YOLO


# Global Configuration & Path Definitions

# Path to the dataset directory containing YOLO-formatted images and labels
DATASET_SRC = Path(
    "/kaggle/input/datasets/alipamda/acnescu-v1i-yolo26-bbox-preprocessed/ACNESCU.v1i.yolo26_bbox_preprocessed"
)

# Path to the pre-trained backbone checkpoint (best.pt or last.pt)
PRETRAINED_WEIGHTS = Path(
    "/kaggle/input/datasets/alipamda/best-v3-2/best_v3.2.pt"
)

# Experiment identifier for output tracking
RUN_NAME = "acne_yolo26_classes_transfer_v1"

# Class mapping matching class_id ordering in .txt label files
CLASS_NAMES = {
    0: "atrophic_scar",
    1: "closed_comedo",
    2: "hypertrophic_scar",
    3: "melasma",
    4: "nevus",
    5: "nodule",
    6: "open_comedo",
    7: "other",
    8: "papule",
    9: "pustule",
}

# Input resolution (higher resolution recommended for detecting fine dermal lesions)
IMGSZ = 1280

EPOCHS = 80
PATIENCE = 20
BATCH = 4

SEED = 42

WORKDIR = Path("/kaggle/working")
DATA_DIR = WORKDIR / "acne_200"
PROJECT_DIR = WORKDIR / "acne_yolo26"
RUNS_DIR = PROJECT_DIR / "runs"
EXPORT_DIR = WORKDIR / "acne_exports"

for directory in (WORKDIR, PROJECT_DIR, RUNS_DIR, EXPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Ultralytics:", __import__("ultralytics").__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("Dataset:", DATASET_SRC)
print("Checkpoint:", PRETRAINED_WEIGHTS)
print("Output:", RUNS_DIR)


# Note: Verify that PRETRAINED_WEIGHTS references the intended .pt model file.
# 3. Dataset Preparation & Configuration

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.182
PyTorch: 2.11.0+cu128
GPU available: True
Device: 0
GPU name: Tesla T4
Dataset: /kaggle/input/datasets/alipamda/acnescu-v1i-yolo26-bbox-preprocessed/ACNESCU.v1i.yolo26_bbox_preprocessed
Checkpoint: /kaggle/input/datasets/alipamda/best-v3-2/best_v3.2.pt
Output: /kaggle/working/acne_yolo26/runs


In [ ]:
# Dataset Preparation & Configuration Setup

if not DATASET_SRC.is_dir():
    raise FileNotFoundError(
        f"Nie znaleziono datasetu: {DATASET_SRC}"
    )

if not PRETRAINED_WEIGHTS.is_file():
    raise FileNotFoundError(
        f"Nie znaleziono checkpointu: {PRETRAINED_WEIGHTS}"
    )

required_dirs = [
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels",
]

missing = [
    p for p in required_dirs
    if not (DATASET_SRC / p).is_dir()
]

if missing:
    raise FileNotFoundError(
        "Brak wymaganych katalogów:\n" + "\n".join(missing)
    )

# Purge existing local dataset directory to prevent stale cached data
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

shutil.copytree(DATASET_SRC, DATA_DIR)

# Generate standardized dataset configuration with explicit pathing and class mappings
data_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": CLASS_NAMES,
    "nc": len(CLASS_NAMES),
}

DATA_YAML = DATA_DIR / "data.yaml"

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False,
        allow_unicode=True,
    )

print(DATA_YAML.read_text())


# Note: If your validation directory is named 'val' instead of 'valid', adjust the paths accordingly in data_config. Ensure 'train', 'val', and 'test' point directly to image directories.
# 4. Pre-Training Annotation Integrity Verification
# Validates YOLO label file formatting, verifies that class indices fall within the target range, and ensures proper image-label file pairing.

path: /kaggle/working/acne_200
train: train/images
val: valid/images
test: test/images
names:
  0: atrophic_scar
  1: closed_comedo
  2: hypertrophic_scar
  3: melasma
  4: nevus
  5: nodule
  6: open_comedo
  7: other
  8: papule
  9: pustule
nc: 10



In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def validate_dataset():
    total_images = 0
    total_objects = {name: 0 for name in CLASS_NAMES.values()}
    errors = []

    for split in ("train", "valid", "test"):
        images_dir = DATA_DIR / split / "images"
        labels_dir = DATA_DIR / split / "labels"

        images = [
            p for p in images_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ]

        label_files = list(labels_dir.rglob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in label_files}

        missing_labels = image_stems - label_stems
        orphan_labels = label_stems - image_stems

        if missing_labels:
            errors.append(
                f"{split}: {len(missing_labels)} images missing corresponding .txt files"
            )

        if orphan_labels:
            errors.append(
                f"{split}: {len(orphan_labels)} .txt files without matching images"
            )

        split_objects = {name: 0 for name in CLASS_NAMES.values()}

        for label_file in label_files:
            for line_no, line in enumerate(
                label_file.read_text().splitlines(), start=1
            ):
                if not line.strip():
                    continue

                try:
                    parts = line.split()

                    if len(parts) != 5:
                        raise ValueError("Exactly 5 values required per annotation line")

                    class_id = int(parts[0])
                    coords = [float(v) for v in parts[1:]]

                    if class_id not in CLASS_NAMES:
                        raise ValueError(
                            f"class_id={class_id} is out of valid range"
                        )

                    if not all(
                        0.0 <= value <= 1.0 for value in coords
                    ):
                        raise ValueError(
                            "Bounding box coordinates fall outside the normalized [0, 1] range"
                        )

                    if coords[2] <= 0 or coords[3] <= 0:
                        raise ValueError(
                            "Bounding box width and height must be strictly positive"
                        )

                    split_objects[CLASS_NAMES[class_id]] += 1

                except Exception as exc:
                    errors.append(
                        f"{label_file}:{line_no}: {exc}"
                    )

        total_images += len(images)

        for name, count in split_objects.items():
            total_objects[name] += count

        print(
            f"{split:5s}: {len(images):4d} images | "
            f"objects: {split_objects}"
        )

    print("\nTotal Images:", total_images)
    print("Total Annotations:", total_objects)

    absent_classes = [
        name for name, count in total_objects.items()
        if count == 0
    ]

    if absent_classes:
        errors.append(
            f"Zero annotations found for classes: {absent_classes}"
        )

    if errors:
        print("\nDATASET VALIDATION ERRORS:")
        for error in errors[:50]:
            print("-", error)

        raise ValueError(
            "Resolve dataset inconsistencies before initiating training."
        )

    print("\nDataset validation successfully completed.")


validate_dataset()


# Note: If validation flags images missing .txt files, investigate carefully. In YOLO detection, an empty label file is valid only for background images containing no objects, which should be an intentional design choice rather than missing annotations.
# 5. Transfer Learning Training Phase
# Core execution block. Loads the baseline pre-trained checkpoint and trains on the updated data.yaml configuration.
# Note: Do not use resume=True here. Because the model architecture adapts to a new multi-class output space, this is treated as a clean transfer learning session rather than a direct continuation of the previous training run.

train:  220 zdjęć | obiekty: {'atrophic_scar': 7004, 'closed_comedo': 4782, 'hypertrophic_scar': 640, 'melasma': 3031, 'nevus': 1151, 'nodule': 172, 'open_comedo': 2526, 'other': 338, 'papule': 4489, 'pustule': 1062}
valid:   27 zdjęć | obiekty: {'atrophic_scar': 892, 'closed_comedo': 687, 'hypertrophic_scar': 75, 'melasma': 556, 'nevus': 137, 'nodule': 18, 'open_comedo': 331, 'other': 40, 'papule': 634, 'pustule': 116}
test :   28 zdjęć | obiekty: {'atrophic_scar': 836, 'closed_comedo': 405, 'hypertrophic_scar': 50, 'melasma': 358, 'nevus': 162, 'nodule': 19, 'open_comedo': 436, 'other': 50, 'papule': 416, 'pustule': 89}

Łącznie zdjęć: 275
Łącznie anotacji: {'atrophic_scar': 8732, 'closed_comedo': 5874, 'hypertrophic_scar': 765, 'melasma': 3945, 'nevus': 1450, 'nodule': 209, 'open_comedo': 3293, 'other': 428, 'papule': 5539, 'pustule': 1267}

Kontrola datasetu zakończona pomyślnie.


In [ ]:
def train_new_classes():
    torch.manual_seed(SEED)
    random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print("Loading pre-trained baseline model:")
    print(PRETRAINED_WEIGHTS)

    model = YOLO(str(PRETRAINED_WEIGHTS))

    print("Source Model Classes:", model.names)
    print("Target Model Classes:", CLASS_NAMES)

    train_params = {
        "data": str(DATA_YAML),

        "project": str(RUNS_DIR),
        "name": RUN_NAME,
        "exist_ok": True,

        # Single GPU or CPU accelerator assignment
        "device": DEVICE,

        "imgsz": IMGSZ,
        "batch": BATCH,
        "epochs": EPOCHS,
        "patience": PATIENCE,

        "optimizer": "AdamW",
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "warmup_epochs": 3.0,
        "cos_lr": True,

        # Moderate data augmentation profile tailored for small datasets
        "mosaic": 0.15,
        "close_mosaic": 10,
        "mixup": 0.0,
        "fliplr": 0.5,
        "flipud": 0.0,
        "degrees": 0.0,
        "translate": 0.05,
        "scale": 0.10,

        "pretrained": True,
        "seed": SEED,
        "deterministic": True,

        "workers": 2,
        "cache": False,
        "amp": torch.cuda.is_available(),

        "val": True,
        "plots": True,
        "verbose": True,
        "save": True,
        "save_period": 10,

        # Clean initialization; not a continuation of previous training history
        "resume": False,
    }

    print("\nTraining Hyperparameters:")
    for key, value in train_params.items():
        print(f"{key:20s}: {value}")

    try:
        results = model.train(**train_params)
        print("\nTraining completed successfully.")
        print("Output Directory:", results.save_dir)
        return results

    finally:
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


results = train_new_classes()


# Note: These hyperparameters serve as a baseline rather than an optimal configuration guarantee. Given the small dataset of 200 images, closely monitor class-wise validation metrics and signs of overfitting. If encountering CUDA Out-Of-Memory (OOM) errors on a standard T4 GPU, reduce the batch size (BATCH) to 2, followed by lowering image size (IMGSZ) to 768 if necessary.
# 6. Model Evaluation on Test Split
# Evaluates the best model checkpoint (best.pt) on the isolated test partition which was excluded from weight updates.

Ładowanie istniejącego modelu:
/kaggle/input/datasets/alipamda/best-v3-2/best_v3.2.pt
Klasy w modelu źródłowym: {0: 'acne'}
Klasy docelowe: {0: 'atrophic_scar', 1: 'closed_comedo', 2: 'hypertrophic_scar', 3: 'melasma', 4: 'nevus', 5: 'nodule', 6: 'open_comedo', 7: 'other', 8: 'papule', 9: 'pustule'}

Parametry treningu:
data                : /kaggle/working/acne_200/data.yaml
project             : /kaggle/working/acne_yolo26/runs
name                : acne_yolo26_classes_transfer_v1
exist_ok            : True
device              : 0
imgsz               : 1280
batch               : 4
epochs              : 80
patience            : 20
optimizer           : AdamW
lr0                 : 0.001
lrf                 : 0.01
weight_decay        : 0.0005
warmup_epochs       : 3.0
cos_lr              : True
mosaic              : 0.15
close_mosaic        : 10
mixup               : 0.0
fliplr              : 0.5
flipud              : 0.0
degrees             : 0.0
translate           : 0.05
scale       

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80      6.63G      2.803      3.449   0.001799        476       1280: 100% ━━━━━━━━━━━━ 55/55 3.0it/s 18.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.6it/s 1.1s0.5s
                   all         27       3486      0.103        0.2     0.0663     0.0202

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/80      6.38G      2.762      3.106   0.001815        329       1280: 100% ━━━━━━━━━━━━ 55/55 3.0it/s 18.1s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.6it/s 1.5s0.7s
                   all         27       3486      0.159      0.176     0.0942     0.0281

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/80      6.46G      2.738      2.913   0.001761        676       1280: 100% ━━━━━━━━━━━━ 55/55 3.0it/s 18.1s0.3s
                 Class     Images  Instance

In [ ]:
def evaluate_test_set():
    best_path = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

    if not best_path.is_file():
        raise FileNotFoundError(
            f"Best checkpoint artifact not found: {best_path}"
        )

    model = YOLO(str(best_path))

    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        plots=True,
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_test",
        exist_ok=True,
    )

    print("\n========== TEST RESULTS ==========")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    print("\n========== PER-CLASS RESULTS ==========")

    for i, name in model.names.items():
        if i >= len(metrics.box.p):
            continue

        print(f"\n{name}")
        print(f"Precision: {metrics.box.p[i]:.4f}")
        print(f"Recall:    {metrics.box.r[i]:.4f}")
        print(f"mAP50:     {metrics.box.ap50[i]:.4f}")
        print(f"mAP50-95:  {metrics.box.ap[i]:.4f}")

    print("\nEvaluation successfully completed.")
    print("Model Artifact:", best_path)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# Execute evaluation pipeline
test_metrics = evaluate_test_set()


# Note: Do not treat high evaluation metrics on a small test partition as definitive proof of real-world generalization robustness. In addition to quantitative metrics, perform qualitative visual validation on inference outputs—paying close attention to potential classification confusion between papules and pustules, as well as the detection accuracy of small comedones.
# 7. Model Export & Artifact Archiving
# Copies model weights and compresses the entire experiment directory into a downloadable ZIP archive for retrieval from Kaggle.

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 120 layers, 9,469,050 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5519.0±845.4 MB/s, size: 3208.7 KB)
val: Scanning /kaggle/working/acne_200/test/labels... 28 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 28/28 938.9it/s 0.0s
val: New cache created: /kaggle/working/acne_200/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.4s0.3ss
                   all         28       2821      0.386      0.334      0.289      0.105
         atrophic_scar         28        836      0.502      0.393      0.364      0.128
         closed_comedo         28        405      0.463      0.249      0.215     0.0647
     hypertrophic_scar         18         50      0.352        0.3      0.278     0.0938
               melasma         25        358      0.456      0.

In [ ]:
run_dir = RUNS_DIR / RUN_NAME
weights_dir = run_dir / "weights"

best_path = weights_dir / "best.pt"
last_path = weights_dir / "last.pt"

if not best_path.is_file():
    raise FileNotFoundError(
        f"Best model checkpoint artifact missing: {best_path}"
    )

shutil.copy2(
    best_path,
    EXPORT_DIR / "acne_yolo26_classes_best.pt"
)

if last_path.is_file():
    shutil.copy2(
        last_path,
        EXPORT_DIR / "acne_yolo26_classes_last.pt"
    )

shutil.copy2(
    DATA_YAML,
    EXPORT_DIR / "data.yaml"
)

# Archive experiment logs, performance curves, and training configuration
archive_path = shutil.make_archive(
    str(EXPORT_DIR / "training_run"),
    "zip",
    root_dir=run_dir,
)

print("Export successfully completed.")
print("Best Model:", EXPORT_DIR / "acne_yolo26_classes_best.pt")
print("Last Model:", EXPORT_DIR / "acne_yolo26_classes_last.pt")
print("Archive Bundle:", archive_path)
print("\nExport Directory Contents:")

for p in sorted(EXPORT_DIR.iterdir()):
    print(f"{p.name}: {p.stat().st_size / 1024**2:.2f} MB")


# Note: In Kaggle, output files are located in /kaggle/working/acne_exports/. Make sure to download your artifacts before the active session terminates.
# 8. Notebook Execution Sequence
# Run the notebook cells in the following order:
# 1. Setup & Configuration — Configure dataset and checkpoint paths.
# 2. Dataset Preparation — Generate the updated data.yaml targeting the new class configuration.
# 3. Annotation Validation — Run integrity checks to catch formatting errors before training.
# 4. Transfer Learning Training — Execute train_new_classes().
# 5. Model Evaluation — Run evaluate_test_set() on the independent test partition.
# 6. Export Artifacts — Bundle and copy model weights and outputs to /kaggle/working/acne_exports/.
# 
# Key Architectural Simplifications Made in This Pipeline:
# - Removed Optuna optimization routines, SQLite databases, and parallel multi-GPU orchestration.
# - Eliminated legacy training resumption workflows dependent on previous state checkpoints.
# - Removed automated hyperparameter grid searches.
# - Streamlined directory structures, bypassing redundant dataset copying steps.
# - Consolidated duplicated logging and evaluation boilerplate.
# 
# Critical Considerations:
# 1. Catastrophic Forgetting / Class Drift: Fine-tuning exclusively on 200 samples may degrade performance on generalized categories absent from the new subset. This is an expected trade-off in target-specific transfer learning without legacy label alignment.
# 2. Unannotated Objects in Training Data: If an image contains multiple lesion instances but only a subset are annotated, unlabeled instances will be penalized as background during loss calculation. Ensure comprehensive annotation coverage wherever possible.
# 3. Small Test Set Variance: Split datasets prior to training and never tune hyperparameters using test metrics. If multiple images originate from the same patient or session, group them into a single split to prevent data leakage.
# 
# Final Recommendation: Initialize training using best.pt from your baseline model, perform a single controlled multi-class fine-tuning run, and evaluate class-specific metrics independently. Avoid mixing the legacy 800 unannotated images into the multi-class training loop without explicit category labels.

Eksport zakończony.
Best model: /kaggle/working/acne_exports/acne_yolo26_classes_best.pt
Last model: /kaggle/working/acne_exports/acne_yolo26_classes_last.pt
Archiwum: /kaggle/working/acne_exports/training_run.zip

Zawartość eksportu:
acne_yolo26_classes_best.pt: 19.47 MB
acne_yolo26_classes_last.pt: 19.47 MB
data.yaml: 0.00 MB
training_run.zip: 751.53 MB
